# Phân tích E-Commerce Pakistan — Spark SQL
**Dataset:** Pakistan Largest E-Commerce Dataset  
**Nguồn:** Kaggle | **Quy mô:** ~905,208 dòng, 26 cột | **Thời gian:** 2016–2018

> Pipeline tiền xử lý được lấy từ `preprocessing.ipynb`.  
> Dữ liệu đã làm sạch được đọc từ HDFS: `hdfs://localhost:9000/ecom/ecom_clean_csv`


## 1. Khởi động Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PakEcommerce_Analysis") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .getOrCreate()

print("Spark version:", spark.version)
print("✅ Spark khởi động thành công!")


Spark version: 3.3.2
✅ Spark khởi động thành công!


## 2. Đọc dữ liệu đã làm sạch từ HDFS

Dữ liệu này đã được `preprocessing.ipynb` xử lý đầy đủ:
- Đổi tên cột sang snake_case
- Xóa cột rác, dòng trắng, dòng trùng lặp
- Ép kiểu số, chuẩn hóa chữ thường cho cột phân loại
- Lọc giá trị không hợp lệ (price ≥ 0, qty > 0)


In [2]:
hdfs_clean_path = "hdfs://localhost:9000/ecom/ecom_clean_csv"

df_clean = spark.read.csv(
    hdfs_clean_path,
    header=True,
    inferSchema=True
)

print(f"✅ Đọc dữ liệu thành công!")
print(f"📊 Số dòng: {df_clean.count():,}")
print(f"📋 Số cột: {len(df_clean.columns)}")
df_clean.printSchema()


✅ Đọc dữ liệu thành công!
📊 Số dòng: 584,238
📋 Số cột: 24
root
 |-- item_id: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- price: double (nullable = true)
 |-- qty_ordered: double (nullable = true)
 |-- grand_total: double (nullable = true)
 |-- increment_id: string (nullable = true)
 |-- category_name_1: string (nullable = true)
 |-- sales_commission_code: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- working_date: string (nullable = true)
 |-- bi_status: string (nullable = true)
 |-- mv: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- customer_since: string (nullable = true)
 |-- m_y: string (nullable = true)
 |-- fy: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_year: string (nullable = true)
 |-- order_month: string (nu

## 3. Tạo các cột bổ sung phục vụ phân tích

Bổ sung `status_clean`, `is_successful`, `quarter` — những cột mà các câu SQL cần
nhưng preprocessing chưa tạo (do preprocessing dùng pipeline riêng).


In [3]:
from pyspark.sql.functions import col, when, quarter as spark_quarter
import pyspark.sql.functions as F

df = df_clean \
    .withColumn(
        "status_clean",
        when(col("status") == "complete", "completed")
        .when(col("status") == "canceled", "canceled")
        .when(col("status").isin("order_refunded", "refund"), "refunded")
        .when(col("status") == "received", "received")
        .otherwise("other")
    ) \
    .withColumn(
        "is_successful",
        when(col("status_clean").isin("completed", "received"), 1).otherwise(0)
    ) \
    .withColumn(
        "quarter",
        spark_quarter(col("created_at"))
    )

# Đổi tên fy → fiscal_year cho nhất quán với SQL bên dưới
if "fy" in df.columns:
    df = df.withColumnRenamed("fy", "fiscal_year")

print("✅ Đã tạo cột bổ sung: status_clean, is_successful, quarter")
df.select("status", "status_clean", "is_successful", "quarter", "order_year", "order_month").show(5)


✅ Đã tạo cột bổ sung: status_clean, is_successful, quarter
+--------+------------+-------------+-------+----------+-----------+
|  status|status_clean|is_successful|quarter|order_year|order_month|
+--------+------------+-------------+-------+----------+-----------+
|canceled|    canceled|            0|   null|      null|       null|
|complete|   completed|            1|   null|      null|       null|
|canceled|    canceled|            0|   null|      null|       null|
|complete|   completed|            1|   null|      null|       null|
|complete|   completed|            1|   null|      null|       null|
+--------+------------+-------------+-------+----------+-----------+
only showing top 5 rows



## 4. Tạo 4 Temp View cho Spark SQL

| View | Mô tả |
|------|-------|
| `orders` | Thông tin đơn hàng, trạng thái, thời gian |
| `order_items` | Chi tiết sản phẩm, giá, số lượng |
| `customers` | Danh sách khách hàng duy nhất |
| `categories` | Tổng hợp theo danh mục |


In [4]:
from pyspark.sql.functions import count, avg, sum as spark_sum

# ── View 1: orders ──────────────────────────────────────────────────────────
orders = df.select(
    col("item_id"),
    col("increment_id").alias("order_id"),
    col("customer_id"),
    col("created_at").alias("order_date"),
    col("order_year").alias("year"),
    col("order_month").alias("month"),
    col("quarter"),
    col("fiscal_year"),
    col("status_clean").alias("status"),
    col("is_successful"),
    col("payment_method"),
    col("bi_status")
)
orders.createOrReplaceTempView("orders")

# ── View 2: order_items ─────────────────────────────────────────────────────
order_items = df.select(
    col("item_id"),
    col("increment_id").alias("order_id"),
    col("sku"),
    col("category_name_1").alias("category"),
    col("price"),
    col("qty_ordered"),
    col("grand_total"),
    col("discount_amount"),
    col("mv").alias("merchandise_value")
)
order_items.createOrReplaceTempView("order_items")

# ── View 3: customers ───────────────────────────────────────────────────────
customers = df.select(
    col("customer_id"),
    col("customer_since")
).dropDuplicates(["customer_id"]) \
 .filter(col("customer_id").isNotNull())
customers.createOrReplaceTempView("customers")

# ── View 4: categories ──────────────────────────────────────────────────────
categories = df.groupBy(
    col("category_name_1").alias("category")
).agg(
    count("item_id").alias("total_items"),
    F.avg("price").alias("avg_price"),
    F.sum("grand_total").alias("total_revenue")
).filter(col("category").isNotNull())
categories.createOrReplaceTempView("categories")

print("✅ Đã tạo 4 Temp View:")
for t in spark.catalog.listTables():
    print(f"   - {t.name} (temporary: {t.isTemporary})")


✅ Đã tạo 4 Temp View:
   - categories (temporary: True)
   - customers (temporary: True)
   - order_items (temporary: True)
   - orders (temporary: True)


## 5. Kiểm tra Temp View

In [5]:
print("=== orders (5 dòng đầu) ===")
spark.sql("SELECT * FROM orders LIMIT 5").show(truncate=True)

print("=== order_items (5 dòng đầu) ===")
spark.sql("SELECT * FROM order_items LIMIT 5").show(truncate=True)

print("=== customers (5 dòng đầu) ===")
spark.sql("SELECT * FROM customers LIMIT 5").show(truncate=True)

print("=== categories (5 dòng đầu) ===")
spark.sql("SELECT * FROM categories LIMIT 5").show(truncate=True)


=== orders (5 dòng đầu) ===
+-------+---------+-----------+----------+----+-----+-------+-----------+---------+-------------+--------------+---------+
|item_id| order_id|customer_id|order_date|year|month|quarter|fiscal_year|   status|is_successful|payment_method|bi_status|
+-------+---------+-----------+----------+----+-----+-------+-----------+---------+-------------+--------------+---------+
| 211293|100147554|         52|  7/1/2016|null| null|   null|       FY17| canceled|            0| ublcreditcard|    gross|
| 212193|100148147|        345|  7/3/2016|null| null|   null|       FY17|completed|            1|           cod|      net|
| 212209|100148155|        352|  7/3/2016|null| null|   null|       FY17| canceled|            0|     mygateway|    gross|
| 212251|100148176|        369|  7/3/2016|null| null|   null|       FY17|completed|            1|           cod|      net|
| 212321|100148213|        395|  7/4/2016|null| null|   null|       FY17|completed|            1|           cod

---
# Truy vấn Spark SQL

## Câu 1: Numerical Summarizations — Thống kê doanh thu theo danh mục sản phẩm
**Kỹ thuật:** GROUP BY + Aggregation (COUNT DISTINCT, SUM, AVG, MIN, MAX, ROUND)  
**Ý nghĩa:** Xác định danh mục đóng góp doanh thu cao nhất và tỷ lệ giảm giá trung bình.


In [6]:
result1 = spark.sql("""
    SELECT 
        oi.category,
        COUNT(DISTINCT o.order_id)          AS total_orders,
        SUM(oi.grand_total)                 AS total_revenue,
        ROUND(AVG(oi.grand_total), 2)       AS avg_order_value,
        MIN(oi.price)                       AS min_price,
        MAX(oi.price)                       AS max_price,
        SUM(oi.discount_amount)             AS total_discount,
        ROUND(SUM(oi.discount_amount) / NULLIF(SUM(oi.grand_total), 0) * 100, 2)
                                            AS discount_rate_pct
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    WHERE oi.category IS NOT NULL
    GROUP BY oi.category
    ORDER BY total_revenue DESC
""")
result1.show(15, truncate=False)


+------------------+------------+--------------------+---------------+---------+---------+--------------------+-----------------+
|category          |total_orders|total_revenue       |avg_order_value|min_price|max_price|total_discount      |discount_rate_pct|
+------------------+------------+--------------------+---------------+---------+---------+--------------------+-----------------+
|mobiles & tablets |107858      |3.2917487896659894E9|20036.94       |0.0      |163000.0 |1.482003965507002E8 |4.5              |
|women's fashion   |38874       |1.9515044755644999E9|10170.87       |0.0      |65000.0  |8.38906418114E7     |4.3              |
|appliances        |47901       |1.1527553050945008E9|13488.2        |0.0      |479000.0 |7.387984096409997E7 |6.41             |
|men's fashion     |67573       |9.021606685315002E8 |4006.58        |0.0      |97180.0  |4.117621618989997E7 |4.56             |
|superstore        |21511       |8.543654041505011E8 |3350.93        |0.0      |63500.0  |

## Câu 2: Filtering — Lọc đơn hàng giá trị cao bị hủy hoặc hoàn trả
**Kỹ thuật:** WHERE + Subquery (AVG làm ngưỡng lọc động) + JOIN  
**Ý nghĩa:** Phát hiện đơn hàng giá trị lớn bị hủy/hoàn để ưu tiên xử lý.


In [7]:
result2 = spark.sql("""
    SELECT 
        o.order_id,
        o.customer_id,
        o.order_date,
        oi.category,
        ROUND(oi.grand_total, 2)            AS grand_total,
        ROUND(oi.discount_amount, 2)        AS discount_amount,
        o.payment_method,
        o.status
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE 
        o.status IN ('refunded', 'canceled')
        AND oi.grand_total > (
            SELECT AVG(grand_total) * 2 FROM order_items
        )
        AND o.order_date IS NOT NULL
    ORDER BY oi.grand_total DESC
""")
result2.show(15, truncate=False)


+---------+-----------+----------+-----------------+-----------+---------------+--------------+--------+
|order_id |customer_id|order_date|category         |grand_total|discount_amount|payment_method|status  |
+---------+-----------+----------+-----------------+-----------+---------------+--------------+--------+
|100323300|5032       |6/8/2017  |mobiles & tablets|1.7888E7   |0.0            |jazzvoucher   |canceled|
|100323300|5032       |6/8/2017  |mobiles & tablets|1.7888E7   |0.0            |jazzvoucher   |canceled|
|100323297|5032       |6/8/2017  |mobiles & tablets|1.7888E7   |0.0            |jazzwallet    |canceled|
|100323300|5032       |6/8/2017  |mobiles & tablets|1.7888E7   |0.0            |jazzvoucher   |canceled|
|100323297|5032       |6/8/2017  |mobiles & tablets|1.7888E7   |0.0            |jazzwallet    |canceled|
|100323297|5032       |6/8/2017  |mobiles & tablets|1.7888E7   |0.0            |jazzwallet    |canceled|
|100323297|5032       |6/8/2017  |mobiles & tablets|1.7

## Câu 3: Projection — Xu hướng doanh thu theo tháng và quý
**Kỹ thuật:** GROUP BY tháng/quý/năm + Window Function SUM OVER (doanh thu lũy kế YTD)  
**Ý nghĩa:** Xác định tháng doanh thu đỉnh điểm và theo dõi tích lũy từng năm.


In [8]:
result3 = spark.sql("""
    SELECT 
        o.year,
        o.quarter,
        o.month,
        COUNT(DISTINCT o.order_id)          AS total_orders,
        ROUND(SUM(oi.grand_total), 2)       AS monthly_revenue,
        ROUND(SUM(SUM(oi.grand_total)) OVER (
            PARTITION BY o.year 
            ORDER BY o.month
        ), 2)                               AS cumulative_revenue_ytd,
        ROUND(AVG(oi.grand_total), 2)       AS avg_order_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
        AND o.year IS NOT NULL
    GROUP BY o.year, o.quarter, o.month
    ORDER BY o.year, o.month
""")
result3.show(36, truncate=False)


+----+-------+-----+------------+---------------+----------------------+---------------+
|year|quarter|month|total_orders|monthly_revenue|cumulative_revenue_ytd|avg_order_value|
+----+-------+-----+------------+---------------+----------------------+---------------+
+----+-------+-----+------------+---------------+----------------------+---------------+



## Câu 4: Join — Phân tích hành vi khách hàng mới vs khách hàng cũ
**Kỹ thuật:** JOIN 3 bảng (orders + order_items + customers) + GROUP BY + CASE WHEN  
**Ý nghĩa:** So sánh mức chi tiêu và tỷ lệ hoàn thành đơn hàng theo nhóm khách hàng.


In [9]:
result4 = spark.sql("""
    SELECT 
        c.customer_since,
        COUNT(DISTINCT o.customer_id)       AS total_customers,
        COUNT(DISTINCT o.order_id)          AS total_orders,
        ROUND(AVG(oi.grand_total), 2)       AS avg_spending,
        ROUND(SUM(oi.grand_total), 2)       AS total_spending,
        ROUND(COUNT(DISTINCT o.order_id) / 
              COUNT(DISTINCT o.customer_id), 2) AS avg_orders_per_customer,
        SUM(CASE WHEN o.status = 'completed' 
                 THEN 1 ELSE 0 END)         AS completed_orders,
        SUM(CASE WHEN o.status = 'canceled' 
                 THEN 1 ELSE 0 END)         AS canceled_orders
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c    ON o.customer_id = c.customer_id
    WHERE c.customer_since IS NOT NULL
    GROUP BY c.customer_since
    ORDER BY c.customer_since
""")
result4.show(20, truncate=False)


+--------------+---------------+------------+------------+---------------+-----------------------+----------------+---------------+
|customer_since|total_customers|total_orders|avg_spending|total_spending |avg_orders_per_customer|completed_orders|canceled_orders|
+--------------+---------------+------------+------------+---------------+-----------------------+----------------+---------------+
|#N/A          |1              |9           |3932.65     |66855.0        |9.0                    |0               |5              |
|2016-10       |2593           |12606       |9449.05     |4.0783982267E8 |4.86                   |18044           |15285          |
|2016-11       |14697          |55756       |4466.55     |1.00793949523E9|3.79                   |92583           |72474          |
|2016-12       |2548           |7799        |8516.34     |1.9555217848E8 |3.06                   |10381           |6922           |
|2016-7        |2406           |44851       |5318.44     |6.2827821149E8 |18

## Câu 5: Window Functions — Xếp hạng danh mục theo doanh thu từng năm
**Kỹ thuật:** RANK() OVER (PARTITION BY year) + LAG() tính tăng trưởng YoY  
**Ý nghĩa:** Top 5 danh mục doanh thu cao nhất từng năm kèm tỷ lệ tăng trưởng YoY%.


In [10]:
result5 = spark.sql("""
    SELECT *
    FROM (
        SELECT 
            o.year,
            oi.category,
            ROUND(SUM(oi.grand_total), 2)   AS category_revenue,
            RANK() OVER (
                PARTITION BY o.year 
                ORDER BY SUM(oi.grand_total) DESC
            )                               AS revenue_rank,
            ROUND(SUM(oi.grand_total) * 100.0 / 
                SUM(SUM(oi.grand_total)) OVER (PARTITION BY o.year), 2
            )                               AS revenue_share_pct,
            ROUND(LAG(SUM(oi.grand_total)) OVER (
                PARTITION BY oi.category ORDER BY o.year
            ), 2)                           AS prev_year_revenue,
            ROUND((SUM(oi.grand_total) - LAG(SUM(oi.grand_total)) OVER (
                PARTITION BY oi.category ORDER BY o.year)
            ) * 100.0 / NULLIF(LAG(SUM(oi.grand_total)) OVER (
                PARTITION BY oi.category ORDER BY o.year), 0), 2
            )                               AS yoy_growth_pct
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE oi.category IS NOT NULL 
            AND o.status = 'completed'
            AND o.year IS NOT NULL
        GROUP BY o.year, oi.category
    )
    WHERE revenue_rank <= 5
    ORDER BY year, revenue_rank
""")
result5.show(30, truncate=False)


+----+--------+----------------+------------+-----------------+-----------------+--------------+
|year|category|category_revenue|revenue_rank|revenue_share_pct|prev_year_revenue|yoy_growth_pct|
+----+--------+----------------+------------+-----------------+-----------------+--------------+
+----+--------+----------------+------------+-----------------+-----------------+--------------+



## Câu 6: Subquery — Xác định khách hàng VIP vượt mức chi tiêu trung bình
**Kỹ thuật:** Nested Subquery 2 tầng + JOIN 3 bảng + GROUP BY  
**Ý nghĩa:** Phân khúc khách hàng cao giá trị để xây dựng chương trình ưu đãi riêng.


In [11]:
result6 = spark.sql("""
    SELECT 
        o.customer_id,
        c.customer_since,
        COUNT(DISTINCT o.order_id)          AS total_orders,
        ROUND(SUM(oi.grand_total), 2)       AS total_spending,
        ROUND(AVG(oi.grand_total), 2)       AS avg_order_value,
        ROUND(SUM(oi.discount_amount), 2)   AS total_discount_received
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c    ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
        AND o.customer_id IN (
            SELECT customer_id
            FROM (
                SELECT customer_id, SUM(grand_total) AS spending
                FROM orders o2
                JOIN order_items oi2 ON o2.order_id = oi2.order_id
                WHERE o2.status = 'completed'
                GROUP BY customer_id
            )
            WHERE spending > (
                SELECT AVG(total) FROM (
                    SELECT SUM(grand_total) AS total
                    FROM order_items
                    GROUP BY order_id
                )
            )
        )
    GROUP BY o.customer_id, c.customer_since
    ORDER BY total_spending DESC
""")
result6.show(15, truncate=False)


+-----------+--------------+------------+--------------+---------------+-----------------------+
|customer_id|customer_since|total_orders|total_spending|avg_order_value|total_discount_received|
+-----------+--------------+------------+--------------+---------------+-----------------------+
|3675       |2016-8        |7           |2.3095975E8   |43692.73       |0.0                    |
|39685      |2017-3        |2           |1.2744921E8   |57591.15       |0.0                    |
|50387      |2017-6        |1           |7.1552E7      |1.7888E7       |0.0                    |
|32794      |2017-2        |3           |3.18579016E7  |43109.47       |0.0                    |
|11508      |2016-10       |13          |2.6450322E7   |59172.98       |9000.0                 |
|68948      |2017-11       |1           |2.0039481E7   |27489.0        |0.0                    |
|28014      |2016-12       |3           |1.9787742E7   |51936.33       |0.0                    |
|7456       |2016-9        |6 

## Câu 7: Group By nâng cao — Tỷ lệ hoàn thành đơn theo phương thức thanh toán
**Kỹ thuật:** GROUP BY + CASE WHEN + Aggregation + tính tỷ lệ %  
**Ý nghĩa:** Tối ưu phương thức thanh toán, giảm tỷ lệ hủy đơn hàng.


In [12]:
result7 = spark.sql("""
    SELECT 
        o.payment_method,
        COUNT(DISTINCT o.order_id)              AS total_orders,
        SUM(CASE WHEN o.status = 'completed' 
                 THEN 1 ELSE 0 END)             AS completed,
        SUM(CASE WHEN o.status = 'canceled'  
                 THEN 1 ELSE 0 END)             AS canceled,
        SUM(CASE WHEN o.status = 'refunded'  
                 THEN 1 ELSE 0 END)             AS refunded,
        ROUND(SUM(CASE WHEN o.status = 'completed' 
                       THEN 1 ELSE 0 END) * 100.0 
              / NULLIF(COUNT(DISTINCT o.order_id), 0), 2) AS completion_rate_pct,
        ROUND(AVG(oi.grand_total), 2)           AS avg_order_value,
        ROUND(SUM(oi.grand_total), 2)           AS total_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.payment_method IS NOT NULL
    GROUP BY o.payment_method
    ORDER BY completion_rate_pct DESC
""")
result7.show(20, truncate=False)


+-----------------+------------+---------+--------+--------+-------------------+---------------+---------------+
|payment_method   |total_orders|completed|canceled|refunded|completion_rate_pct|avg_order_value|total_revenue  |
+-----------------+------------+---------+--------+--------+-------------------+---------------+---------------+
|jazzwallet       |21613       |49303    |53905   |2389    |228.12             |3819.31        |4.9281728746E8 |
|cod              |182342      |326079   |65626   |101291  |178.83             |5019.73        |3.54681947566E9|
|customercredit   |5119        |8613     |109     |2969    |168.26             |12.63          |250816.24      |
|financesettlement|10          |15       |1       |9       |150.00             |113701.48      |2842537.0      |
|cashatdoorstep   |644         |910      |7       |74      |141.30             |14175.12       |1.4203472E7    |
|productcredit    |100         |125      |10      |76      |125.00             |0.0            |

## Câu 8: Time Series — Tăng trưởng doanh thu tháng qua tháng (MoM)
**Kỹ thuật:** LAG() + SUM() OVER (ROWS BETWEEN) + Subquery  
**Ý nghĩa:** Phát hiện tháng đột biến doanh thu, hỗ trợ dự báo xu hướng.


In [13]:
result8 = spark.sql("""
    SELECT 
        year,
        month,
        ROUND(monthly_revenue, 2)               AS monthly_revenue,
        ROUND(LAG(monthly_revenue) OVER (
            ORDER BY year, month
        ), 2)                                   AS prev_month_revenue,
        ROUND((monthly_revenue - LAG(monthly_revenue) OVER (
            ORDER BY year, month)
        ) * 100.0 / NULLIF(LAG(monthly_revenue) OVER (
            ORDER BY year, month), 0), 2)       AS mom_growth_pct,
        ROUND(SUM(monthly_revenue) OVER (
            PARTITION BY year
            ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2)                                   AS cumulative_revenue
    FROM (
        SELECT 
            o.year,
            o.month,
            SUM(oi.grand_total)                 AS monthly_revenue,
            COUNT(DISTINCT o.order_id)          AS total_orders
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
            AND o.year IS NOT NULL
        GROUP BY o.year, o.month
    )
    ORDER BY year, month
""")
result8.show(36, truncate=False)


+----+-----+---------------+------------------+--------------+------------------+
|year|month|monthly_revenue|prev_month_revenue|mom_growth_pct|cumulative_revenue|
+----+-----+---------------+------------------+--------------+------------------+
+----+-----+---------------+------------------+--------------+------------------+



## Câu 9: Window Functions — Top 3 SKU bán chạy nhất trong từng danh mục
**Kỹ thuật:** RANK() OVER (PARTITION BY category) + % đóng góp doanh thu  
**Ý nghĩa:** Quản lý tồn kho và ưu tiên quảng bá sản phẩm chủ lực.


In [14]:
result9 = spark.sql("""
    SELECT *
    FROM (
        SELECT
            oi.category,
            oi.sku,
            COUNT(DISTINCT o.order_id)          AS total_orders,
            ROUND(SUM(oi.grand_total), 2)       AS sku_revenue,
            RANK() OVER (
                PARTITION BY oi.category
                ORDER BY SUM(oi.grand_total) DESC
            )                                   AS rank_in_category,
            ROUND(SUM(oi.grand_total) * 100.0 /
                SUM(SUM(oi.grand_total)) OVER (
                    PARTITION BY oi.category
                ), 2)                           AS revenue_share_in_category_pct,
            ROUND(AVG(oi.price), 2)             AS avg_price,
            SUM(oi.qty_ordered)                 AS total_qty_sold
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        WHERE o.status = 'completed'
            AND oi.category IS NOT NULL
            AND oi.sku IS NOT NULL
        GROUP BY oi.category, oi.sku
    )
    WHERE rank_in_category <= 3
    ORDER BY category, rank_in_category
""")
result9.show(30, truncate=False)


+-----------------+-------------------------------------------+------------+-----------+----------------+-----------------------------+---------+--------------+
|category         |sku                                        |total_orders|sku_revenue|rank_in_category|revenue_share_in_category_pct|avg_price|total_qty_sold|
+-----------------+-------------------------------------------+------------+-----------+----------------+-----------------------------+---------+--------------+
|\n               |Infinix Hot 4 Nationwide-Gold              |304         |4346187.0  |1               |20.37                        |12599.0  |329.0         |
|\n               |Infinix Hot 4 Nationwide-Black             |227         |3263989.0  |2               |15.3                         |12599.0  |246.0         |
|\n               |Infinix Hot 4 Hazir-Black-Karachi          |103         |1360692.0  |3               |6.38                         |12599.0  |108.0         |
|appliances       |APPGRE5A81C19C1

## Câu 10: Join + Window Functions — Hiệu suất danh mục so với trung bình toàn sàn
**Kỹ thuật:** JOIN 3 bảng + AVG() OVER (PARTITION BY year) + CASE WHEN  
**Ý nghĩa:** Đánh giá danh mục nào vượt trội hay tụt hậu so với mặt bằng chung.


In [15]:
result10 = spark.sql("""
    SELECT 
        o.year,
        oi.category,
        ROUND(SUM(oi.grand_total), 2)           AS category_revenue,
        COUNT(DISTINCT o.order_id)              AS total_orders,
        ROUND(AVG(oi.grand_total), 2)           AS avg_order_value,
        ROUND(AVG(SUM(oi.grand_total)) OVER (
            PARTITION BY o.year
        ), 2)                                   AS avg_revenue_all_categories,
        ROUND(SUM(oi.grand_total) - AVG(SUM(oi.grand_total)) OVER (
            PARTITION BY o.year
        ), 2)                                   AS revenue_vs_avg,
        CASE 
            WHEN SUM(oi.grand_total) > AVG(SUM(oi.grand_total)) OVER (
                PARTITION BY o.year) 
            THEN 'Trên trung bình'
            ELSE 'Dưới trung bình'
        END                                     AS performance_flag,
        cat.total_items                         AS total_items_in_category
    FROM orders o
    JOIN order_items oi  ON o.order_id = oi.order_id
    JOIN categories cat  ON oi.category = cat.category
    WHERE o.status = 'completed'
        AND oi.category IS NOT NULL
        AND o.year IS NOT NULL
    GROUP BY o.year, oi.category, cat.total_items
    ORDER BY o.year, category_revenue DESC
""")
result10.show(30, truncate=False)


+----+--------+----------------+------------+---------------+--------------------------+--------------+----------------+-----------------------+
|year|category|category_revenue|total_orders|avg_order_value|avg_revenue_all_categories|revenue_vs_avg|performance_flag|total_items_in_category|
+----+--------+----------------+------------+---------------+--------------------------+--------------+----------------+-----------------------+
+----+--------+----------------+------------+---------------+--------------------------+--------------+----------------+-----------------------+

